# P2 · N3 — Parse, Score, and Failure-Code

**Paper 2 — MARQ-Bench: Machine-Authored Data Quality**

Turns the raw generations from N2 into measured results.

1. Parse every raw response into the rule IR
2. Execute every rule on the **evaluation split** and measure retention
3. Assign the pre-specified failure codes F1–F10
4. Compute set-level retention and the rule-effect correlation matrix
5. Summarise by corpus, condition, and model

**Prerequisites:** N0, N1, and N2 complete. Reads `runs/runs.jsonl`.

**Nothing here touches the authoring side.** N2 captured raw text and stopped;
this notebook does all parsing and scoring. That separation is the anti-tautology
boundary made visible.

**Runtime:** 10–20 minutes on a full sweep, dominated by loading the corpora.
Rule masks are cached by canonical form, so identical rules across seeds and
conditions are evaluated once.

## 1 · Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Setup

In [ ]:
import sys, json, datetime, itertools
from pathlib import Path
import pandas as pd
import numpy as np

ROOT        = Path('/content/drive/MyDrive/Paper2_RuleAuthorship')
MODULES     = ROOT / 'Notebooks'
CHECKPOINTS = ROOT / 'checkpoints'
ARTIFACTS   = ROOT / 'artifacts'
RUNS        = ROOT / 'runs'

assert ROOT.exists() and MODULES.exists()
ARTIFACTS.mkdir(parents=True, exist_ok=True)
if str(MODULES) not in sys.path:
    sys.path.insert(0, str(MODULES))

import llmauth_census as C
import llmauth_ir as IR
import llmauth_checkpoint as CK

ckpt = CK.Checkpoint(CHECKPOINTS)
RUN_LOG_PATH = RUNS / 'runs.jsonl'

PATTERNS = {
    'bank_marketing':   ['bankfull', 'bank-full', 'bank_full'],
    'diabetes_130us':   ['diabetic_data', 'diabetes'],
    'online_retail_ii': ['online_retail', 'online retail', 'retail'],
    'nyc_tlc_yellow':   ['yellow_tripdata'],
}

def discover_data():
    files = [p for p in ROOT.rglob('*')
             if p.is_file() and p.suffix.lower() in ('.csv','.parquet','.xlsx')]
    out = {}
    for corpus, frags in PATTERNS.items():
        hits = [p for p in files if any(f in p.name.lower() for f in frags)]
        hits.sort(key=lambda p: p.stat().st_size, reverse=True)
        if hits: out[corpus] = hits[0]
    return out

print('census', C.CENSUS_VERSION, '| ir', IR.SCHEMA_VERSION,
      '| checkpoint', CK.CHECKPOINT_VERSION)

census 1.0.0 | ir 1.0.0 | checkpoint 1.0.0


## 3 · Load the run log

Excludes any mock-backend runs — those are plumbing tests, not data.

In [ ]:
EXCLUDE_MODELS = {'mock-1'}

records = [json.loads(l) for l in open(RUN_LOG_PATH) if l.strip()]
records = [r for r in records if r['key']['model_id'] not in EXCLUDE_MODELS]

print(f'{len(records)} runs after excluding {EXCLUDE_MODELS or "nothing"}\n')
by = {}
for r in records:
    k = (r['key']['corpus_id'], r['key']['model_id'])
    b = by.setdefault(k, {'n':0,'ok':0})
    b['n'] += 1; b['ok'] += bool(r['ok'])
print(f'{"corpus":<18}{"model":<30}{"runs":>6}{"ok":>5}')
print('-'*61)
for (corpus, model), b in sorted(by.items()):
    print(f'{corpus:<18}{model:<30}{b["n"]:>6}{b["ok"]:>5}')

480 runs after excluding {'mock-1'}

corpus            model                           runs   ok
-------------------------------------------------------------
bank_marketing    claude-haiku-4-5-20251001         40   40
bank_marketing    claude-sonnet-4-5-20250929        40   40
bank_marketing    gemini-flash                      40   40
diabetes_130us    claude-haiku-4-5-20251001         40   40
diabetes_130us    claude-sonnet-4-5-20250929        40   40
diabetes_130us    gemini-flash                      40   40
nyc_tlc_yellow    claude-haiku-4-5-20251001         40   40
nyc_tlc_yellow    claude-sonnet-4-5-20250929        40   40
nyc_tlc_yellow    gemini-flash                      40   40
online_retail_ii  claude-haiku-4-5-20251001         40   40
online_retail_ii  claude-sonnet-4-5-20250929        40   40
online_retail_ii  gemini-flash                      40   40


## 4 · Parse into the rule IR

Failed runs are retained as empty rule sets. Unparseable rules are kept and
coded F7. Nothing is dropped — a dropped record would quietly improve every
rate statistic that follows.

In [ ]:
rulesets = {}
for r in records:
    k = r['key']
    key = (k['corpus_id'], k['condition'], k['model_id'], k['seed'], k.get('variant','main'))
    rs = IR.parse_llm_response(
        r['raw_response'] or '',
        corpus_id=k['corpus_id'], condition=k['condition'],
        model_id=k['model_id'], seed=k['seed'],
        temperature=k['temperature'],
        generated_at_utc=r.get('finished_at_utc',''),
        prompt_sha256=r.get('prompt_sha256',''),
    )
    rulesets[key] = rs

n_rules = sum(len(rs) for rs in rulesets.values())
n_empty = sum(1 for rs in rulesets.values() if len(rs) == 0)
n_err   = sum(1 for rs in rulesets.values() if rs.parse_errors)
print(f'{len(rulesets)} rule sets, {n_rules:,} rules total')
print(f'  empty rule sets : {n_empty}')
print(f'  parse errors    : {n_err}')

480 rule sets, 8,617 rules total
  empty rule sets : 0
  parse errors    : 0


## 5 · Load corpora and evaluation splits

Rules are scored on the **evaluation split** — the 80% the census never saw.

In [ ]:
DATA_PATHS = discover_data()
needed = sorted({k[0] for k in rulesets})

evaluation, schemas = {}, {}
for corpus in needed:
    df, _ = C.load_corpus(corpus, DATA_PATHS[corpus])
    split = ckpt.step(f'split_{corpus}', 'json',
                      lambda: (_ for _ in ()).throw(RuntimeError('run N0 first')))[0]
    ev = df.loc[split['evaluation_index']]
    evaluation[corpus] = ev

    cen = ckpt.step(f'census_{corpus}', 'json',
                    lambda: (_ for _ in ()).throw(RuntimeError('run N1 first')))[0]
    schemas[corpus] = IR.CorpusSchema(
        corpus_id=corpus,
        columns={c['name']: IR.ColumnProfile(
            name=c['name'], dtype=c['dtype'],
            null_rate=c['null_rate'] or 0.0,
            distinct_count=c['distinct_count'],
            min_value=c['min_value'], max_value=c['max_value'],
            observed_levels=[v for v, _ in c['top_levels']],
        ) for c in cen['columns']},
    )
    print(f'{corpus:<18} evaluation {len(ev):>9,} rows, {len(schemas[corpus].columns):>2} columns')

  [cached] split_bank_marketing.json  (written 2026-08-08T03:39:10+00:00)
  [cached] census_bank_marketing.json  (written 2026-08-08T15:17:55+00:00)
bank_marketing     evaluation    36,169 rows, 17 columns
  [cached] split_diabetes_130us.json  (written 2026-08-08T03:39:11+00:00)
  [cached] census_diabetes_130us.json  (written 2026-08-08T15:17:58+00:00)
diabetes_130us     evaluation    81,413 rows, 50 columns
  [cached] split_nyc_tlc_yellow.json  (written 2026-08-08T03:39:14+00:00)
  [cached] census_nyc_tlc_yellow.json  (written 2026-08-08T15:18:02+00:00)
nyc_tlc_yellow     evaluation   800,000 rows, 20 columns
  [cached] split_online_retail_ii.json  (written 2026-08-08T14:34:08+00:00)
  [cached] census_online_retail_ii.json  (written 2026-08-08T15:18:01+00:00)
online_retail_ii   evaluation   853,897 rows,  8 columns


## 6 · Evaluate every rule

Masks are cached by canonical form, so a rule that recurs across seeds and
conditions is executed once. On a full sweep this is the difference between
minutes and hours.

In [ ]:
mask_cache = {}
cache_hits = cache_misses = 0

def rule_mask(corpus, rule):
    """Boolean Series, True where the record PASSES. Cached by canonical form."""
    global cache_hits, cache_misses
    ck_ = (corpus, IR.canonical_form(rule))
    if ck_ in mask_cache:
        cache_hits += 1
        return mask_cache[ck_]
    cache_misses += 1
    try:
        m = IR.to_pandas_mask(rule)(evaluation[corpus])
    except Exception:
        m = None            # non-executable against this data
    mask_cache[ck_] = m
    return m

retentions = {}     # (setkey) -> {rule_id: retention}
for i, (key, rs) in enumerate(rulesets.items(), 1):
    corpus = key[0]
    n = len(evaluation[corpus])
    per_rule = {}
    for rule in rs.rules:
        if not rule.is_executable or rule.column not in evaluation[corpus].columns:
            continue
        m = rule_mask(corpus, rule)
        if m is not None:
            per_rule[rule.rule_id] = float(m.sum()) / n
    retentions[key] = per_rule
    if i % 50 == 0 or i == len(rulesets):
        print(f'  {i}/{len(rulesets)} rule sets   cache {cache_hits} hits / {cache_misses} misses')

print(f'\n{len(mask_cache)} distinct rules evaluated for {sum(len(r) for r in retentions.values()):,} rule instances')

  50/480 rule sets   cache 907 hits / 71 misses
  100/480 rule sets   cache 2101 hits / 252 misses
  150/480 rule sets   cache 4261 hits / 342 misses
  200/480 rule sets   cache 5465 hits / 408 misses
  250/480 rule sets   cache 6327 hits / 511 misses
  300/480 rule sets   cache 6789 hits / 581 misses
  350/480 rule sets   cache 7201 hits / 612 misses
  400/480 rule sets   cache 7509 hits / 624 misses
  450/480 rule sets   cache 7845 hits / 636 misses
  480/480 rule sets   cache 7948 hits / 654 misses

654 distinct rules evaluated for 8,594 rule instances


## 7 · Assign failure codes

The pre-specified taxonomy F1–F10, fixed before any output was inspected and
assigned programmatically rather than by judgement.

In [ ]:
for key, rs in rulesets.items():
    IR.code_ruleset(rs, schemas[key[0]], retentions.get(key, {}),
                    assign_vacuous=False)   # natural data: see inert rate below

from collections import Counter
counts = Counter()
for rs in rulesets.values():
    for rule in rs.rules:
        counts.update(c.value for c in rule.failure_codes)

DESC = {'F1':'hallucinated column','F2':'hallucinated category',
        'F3':'contradicts census','F4':'over-tight (<50%)','F5':'vacuous (100%)',
        'F6':'type mismatch','F7':'non-executable','F8':'redundant',
        'F9':'sentinel misread','F10':'fairness-hazardous'}
inerts = [IR.inert_rate(rs, retentions.get(k, {}))
          for k, rs in rulesets.items()]
total_rules = sum(len(rs) for rs in rulesets.values())
print(f'{"code":<6}{"description":<26}{"count":>7}{"% of rules":>12}')
print('-'*51)
for c in [f'F{i}' for i in range(1,11)]:
    n = counts.get(c,0)
    print(f'{c:<6}{DESC[c]:<26}{n:>7}{n/total_rules*100:>11.1f}%')
print()
print(f'F5 is not assigned on natural data (see llmauth_ir). Reported instead:')
print(f'  inert rate: {np.mean(inerts):.1%} of executable rules reject nothing')

code  description                 count  % of rules
---------------------------------------------------
F1    hallucinated column             0        0.0%
F2    hallucinated category        1055       12.2%
F3    contradicts census              0        0.0%
F4    over-tight (<50%)             205        2.4%
F5    vacuous (100%)                  0        0.0%
F6    type mismatch                  97        1.1%
F7    non-executable                 15        0.2%
F8    redundant                       1        0.0%
F9    sentinel misread              310        3.6%
F10   fairness-hazardous            182        2.1%

F5 is not assigned on natural data (see llmauth_ir). Reported instead:
  inert rate: 56.2% of executable rules reject nothing


## 8 · Set-level retention

What fraction of the evaluation split survives the whole rule set. This is the
headline number: a gate that reports full compliance while retaining 20% of the
data has not improved quality, it has destroyed it.

In [ ]:
rows = []
for key, rs in rulesets.items():
    corpus, cond, model, seed, variant = key
    ev = evaluation[corpus]
    keep = pd.Series(True, index=ev.index)
    keep_clean = pd.Series(True, index=ev.index)   # excluding type-mismatched rules
    n_typemismatch = 0
    for rule in rs.rules:
        if not rule.is_executable or rule.column not in ev.columns:
            continue
        m = rule_mask(corpus, rule)
        if m is None:
            continue
        keep &= m
        if IR.FailureCode.F6_TYPE_MISMATCH in rule.failure_codes:
            n_typemismatch += 1
        else:
            keep_clean &= m
    per = retentions.get(key, {})
    rows.append({
        'corpus': corpus, 'condition': cond, 'model': model, 'seed': seed,
        'n_rules': len(rs), 'n_executable': len(rs.executable),
        'set_retention': float(keep.mean()),
        'set_retention_ex_typemismatch': float(keep_clean.mean()),
        'n_typemismatch': n_typemismatch,
        'inert_rate': IR.inert_rate(rs, per),
        'product_of_rule_retentions': float(np.prod(list(per.values()))) if per else 1.0,
        **{c: sum(1 for r in rs.rules if IR.FailureCode(c) in r.failure_codes)
           for c in [f'F{i}' for i in range(1,11)]},
    })

scored = pd.DataFrame(rows)
summary = (scored.groupby(['corpus','condition','model'])
           .agg(runs=('seed','count'), rules=('n_rules','mean'),
                retention=('set_retention','mean'),
                retention_sd=('set_retention','std'),
                F9=('F9','mean'), F4=('F4','mean'), F2=('F2','mean'))
           .round(4))
print(summary.to_string())

                                                       runs  rules  retention  retention_sd   F9   F4    F2
corpus           condition model                                                                           
bank_marketing   A2        claude-haiku-4-5-20251001     10   16.8     0.8773        0.2730  0.3  0.1   3.2
                           claude-sonnet-4-5-20250929    10   20.6     0.6899        0.3711  0.0  0.2   1.7
                           gemini-flash                  10    7.6     0.8184        0.3395  0.0  0.2   0.0
                 A3        claude-haiku-4-5-20251001     10   17.1     0.8997        0.3161  0.0  0.1   0.0
                           claude-sonnet-4-5-20250929    10   24.8     0.4998        0.4943  0.0  0.6   0.0
                           gemini-flash                  10    9.1     0.3551        0.4137  0.0  0.8   0.0
                 A4        claude-haiku-4-5-20251001     10   17.5     0.7629        0.4089  0.0  0.2   0.0
                           c

## 9 · The information ladder

Mean set retention by condition. This is the paper's central table.

In [ ]:
piv = (scored.pivot_table(index=['corpus','model'], columns='condition',
                          values='set_retention', aggfunc='mean')
        .round(4))
print('SET RETENTION\n'); print(piv.to_string()); print()

piv9 = (scored.pivot_table(index=['corpus','model'], columns='condition',
                           values='F9', aggfunc='mean').round(2))
print('MEAN F9 (sentinel misread) RULES PER SET\n'); print(piv9.to_string())

SET RETENTION

condition                                        A2      A3      A4      A5
corpus           model                                                     
bank_marketing   claude-haiku-4-5-20251001   0.8773  0.8997  0.7629  0.1816
                 claude-sonnet-4-5-20250929  0.6899  0.4998  0.6181  0.2366
                 gemini-flash                0.8184  0.3551  0.0185  0.0187
diabetes_130us   claude-haiku-4-5-20251001   0.0000  0.2261  0.9557  0.8804
                 claude-sonnet-4-5-20250929  0.7017  0.2632  0.8500  0.9608
                 gemini-flash                1.0000  0.8627  0.8750  0.7913
nyc_tlc_yellow   claude-haiku-4-5-20251001   0.4398  0.7459  0.7099  0.7236
                 claude-sonnet-4-5-20250929  0.6490  0.7475  0.7529  0.7587
                 gemini-flash                0.7539  0.7603  0.7542  0.7594
online_retail_ii claude-haiku-4-5-20251001   0.7541  0.0000  0.5194  0.6992
                 claude-sonnet-4-5-20250929  0.2300  0.2135  0.0000  0.00

## 9b · How much of the collapse is a type-mismatch artefact?

Models express date constraints as bare epoch integers without stating units.
The IR now interprets those by magnitude, but a rule whose predicate genuinely
does not fit its column's type (`F6`) still rejects every row.

`set_retention` counts every rule as written — the honest production outcome,
since a deployed rule really would reject everything. `set_retention_ex_typemismatch`
excludes F6-coded rules, isolating the rules that failed on their *semantics*
rather than their types. Report both; the gap between them is the size of the
artefact.

In [ ]:
cmp = (scored.groupby(['corpus','condition'])
       [['set_retention','set_retention_ex_typemismatch','n_typemismatch']]
       .mean().round(4))
cmp['gap'] = (cmp.set_retention_ex_typemismatch - cmp.set_retention).round(4)
print(cmp.to_string())

n0  = int((scored.set_retention == 0).sum())
n0c = int((scored.set_retention_ex_typemismatch == 0).sum())
print(f'\nruns at 0% retention           : {n0}/{len(scored)}')
print(f'runs at 0% excluding F6 rules  : {n0c}/{len(scored)}')
print(f'attributable to type mismatch  : {n0 - n0c}')
print(f'\nmean inert rate: {scored.inert_rate.mean():.1%} of executable rules reject nothing')

                            set_retention  set_retention_ex_typemismatch  n_typemismatch     gap
corpus           condition                                                                      
bank_marketing   A2                0.7952                         0.7952          0.0000  0.0000
                 A3                0.5849                         0.5849          0.0000  0.0000
                 A4                0.4665                         0.4665          0.0000  0.0000
                 A5                0.1456                         0.1456          0.0000  0.0000
diabetes_130us   A2                0.5672                         0.5672          0.0000  0.0000
                 A3                0.4507                         0.4507          0.0000  0.0000
                 A4                0.8935                         0.8935          0.0000  0.0000
                 A5                0.8775                         0.8775          0.0000  0.0000
nyc_tlc_yellow   A2           

## 10 · Rule-effect correlation

Registered after C4 revealed that `RatecodeID` nulls, `payment_type = 0`, and
`passenger_count` nulls fall on exactly the same 955,371 rows.

Retention accounting assumes rules remove roughly independent record sets. The
ratio below tests that: near 1.0 means independent effects, well above 1.0 means
rules are removing the same rows and stacking them buys nothing.

In [ ]:
scored['independence_ratio'] = (scored['set_retention'] /
                                scored['product_of_rule_retentions'].replace(0, np.nan))
ind = (scored.groupby(['corpus','condition'])['independence_ratio']
       .mean().unstack().round(3))
print('SET RETENTION / PRODUCT OF PER-RULE RETENTIONS\n')
print(ind.to_string())
print('\n1.0 = independent effects;  >1.0 = rules overlap (same rows rejected)')

SET RETENTION / PRODUCT OF PER-RULE RETENTIONS

condition            A2     A3     A4     A5
corpus                                      
bank_marketing    0.980  0.901  0.958  0.692
diabetes_130us    0.953  1.047  1.001  0.998
nyc_tlc_yellow    3.490  2.524  2.671  2.393
online_retail_ii  0.893  1.009  0.670  0.645

1.0 = independent effects;  >1.0 = rules overlap (same rows rejected)


## 11 · Stability across seeds

Jaccard similarity of canonicalised rule sets between seeds within a cell.
Numerically-equal values are normalised first, so `1` and `1.0` do not count as
a threshold change.

In [ ]:
stab = []
cells_ = {}
for key, rs in rulesets.items():
    cells_.setdefault(key[:3] + (key[4],), []).append(rs)

for cell, sets in sorted(cells_.items()):
    if len(sets) < 2:
        continue
    js = [IR.ruleset_jaccard(a, b) for a, b in itertools.combinations(sets, 2)]
    stab.append({'corpus': cell[0], 'condition': cell[1], 'model': cell[2],
                 'n_seeds': len(sets), 'jaccard_mean': float(np.mean(js)),
                 'jaccard_min': float(np.min(js))})

stability = pd.DataFrame(stab)
if len(stability):
    print(stability.pivot_table(index=['corpus','model'], columns='condition',
                                values='jaccard_mean').round(3).to_string())
    print(f'\noverall mean Jaccard: {stability.jaccard_mean.mean():.3f}')
    print('H3 predicts < 0.80 in a majority of cells:',
          f"{(stability.jaccard_mean < 0.80).mean():.0%} of cells below 0.80")
else:
    print('need at least 2 seeds per cell for stability')

condition                                       A2     A3     A4     A5
corpus           model                                                 
bank_marketing   claude-haiku-4-5-20251001   0.697  0.967  0.908  0.840
                 claude-sonnet-4-5-20250929  0.483  0.676  0.805  0.690
                 gemini-flash                0.565  0.618  0.524  0.589
diabetes_130us   claude-haiku-4-5-20251001   0.300  0.848  0.729  0.629
                 claude-sonnet-4-5-20250929  0.806  0.756  0.632  0.770
                 gemini-flash                0.616  0.563  0.259  0.472
nyc_tlc_yellow   claude-haiku-4-5-20251001   0.529  0.885  0.628  0.387
                 claude-sonnet-4-5-20250929  0.716  0.785  0.358  0.396
                 gemini-flash                0.429  0.566  0.553  0.532
online_retail_ii claude-haiku-4-5-20251001   0.747  0.466  0.273  0.286
                 claude-sonnet-4-5-20250929  0.504  0.578  0.803  0.567
                 gemini-flash                0.473  0.291  0.461

## 11b · Diagnose total-collapse rule sets

Any rule set retaining 0% has an unsatisfiable rule in it. This finds the
culprit by removing rules one at a time and reporting which single rule is
responsible for the collapse.

In [ ]:
collapsed = scored[scored.set_retention == 0.0]
print(f'{len(collapsed)} rule set(s) retain 0% of the evaluation split\n')

seen = set()
for _, row in collapsed.iterrows():
    key = (row.corpus, row.condition, row.model, row.seed, 'main')
    if key not in rulesets:
        continue
    sig = (row.corpus, row.condition, row.model)
    if sig in seen:
        continue
    seen.add(sig)
    rs = rulesets[key]
    ev = evaluation[row.corpus]
    print(f'--- {row.corpus} / {row.condition} / seed {row.seed} ---')
    for rule in rs.rules:
        if not rule.is_executable or rule.column not in ev.columns:
            continue
        m = rule_mask(row.corpus, rule)
        if m is None:
            continue
        r = float(m.mean())
        if r < 0.99:
            print(f'  retains {r:>7.4f}  {rule.column:<22} {rule.predicate_type.value:<13} '
                  f'{json.dumps(rule.parameters)[:70]}')
            if rule.rationale:
                print(f'                 "{rule.rationale[:90]}"')
    print()

if not len(collapsed):
    print('No total collapses.')

77 rule set(s) retain 0% of the evaluation split

--- bank_marketing / A2 / seed 8 ---
  retains  0.8165  previous               cross_column  {"other_column": "pdays", "op": ">"}
                 "If previous contacts is greater than zero, pdays should not be -1 (never contacted), indic"
  retains  0.1835  pdays                  cross_column  {"other_column": "previous", "op": ">="}
                 "If pdays is -1 (never contacted), previous contacts should be 0, otherwise the contact his"

--- bank_marketing / A3 / seed 2 ---
  retains  0.0004  previous               cross_column  {"other_column": "pdays", "op": "=="}
                 "When previous is 0 (no prior contacts), pdays must also be -1 (not previously contacted), "
  retains  0.1831  pdays                  cross_column  {"other_column": "previous", "op": ">"}
                 "When pdays is greater than -1 (client was previously contacted), previous must be greater "

--- bank_marketing / A3 / seed 8 ---
  retains  0.0000

## 12 · Save results

In [ ]:
def build_scored():
    return json.loads(scored.to_json(orient='records'))

_, _ = ckpt.step('n3_scored_rulesets', 'json', build_scored,
                 code_version=IR.SCHEMA_VERSION, force=True)

def build_rules_corpus():
    out = []
    for key, rs in rulesets.items():
        for rule in rs.rules:
            d = rule.to_dict()
            d.update({'corpus': key[0], 'condition': key[1], 'model': key[2],
                      'seed': key[3], 'variant': key[4],
                      'retention': retentions.get(key, {}).get(rule.rule_id)})
            out.append(d)
    return out

_, _ = ckpt.step('n3_rule_corpus', 'json', build_rules_corpus,
                 code_version=IR.SCHEMA_VERSION, force=True)

scored.to_csv(ARTIFACTS / 'N3_scored_rulesets.csv', index=False)
if len(stability):
    stability.to_csv(ARTIFACTS / 'N3_stability.csv', index=False)
print('wrote', ARTIFACTS / 'N3_scored_rulesets.csv')
print(f'\n{sum(len(rs) for rs in rulesets.values()):,} coded rules saved as the rule corpus')
print('\nN3 complete. Next: N4 — downstream models, fairness, and cost.')

  [build ] n3_scored_rulesets.json ...
  [saved ] n3_scored_rulesets.json  (243,773 bytes)
  [build ] n3_rule_corpus.json ...
  [saved ] n3_rule_corpus.json  (4,695,903 bytes)
wrote /content/drive/MyDrive/Paper2_RuleAuthorship/artifacts/N3_scored_rulesets.csv

8,617 coded rules saved as the rule corpus

N3 complete. Next: N4 — downstream models, fairness, and cost.
